# Lab 5


Matrix Representation: In this lab you will be creating a simple linear algebra system. In memory, we will represent matrices as nested python lists as we have done in lecture. In the exercises below, you are required to explicitly test every feature you implement, demonstrating it works.

1. Create a `matrix` class with the following properties:
    * It can be initialized in 2 ways:
        1. with arguments `n` and `m`, the size of the matrix. A newly instanciated matrix will contain all zeros.
        2. with a list of lists of values. Note that since we are using lists of lists to implement matrices, it is possible that not all rows have the same number of columns. Test explicitly that the matrix is properly specified.
    * Matrix instances `M` can be indexed with `M[i][j]` and `M[i,j]`.
    * Matrix assignment works in 2 ways:
        1. If `M_1` and `M_2` are `matrix` instances `M_1=M_2` sets the values of `M_1` to those of `M_2`, if they are the same size. Error otherwise.
        2. In example above `M_2` can be a list of lists of correct size.


In [6]:
class matrix:
    def __init__(self, *args):
        if len(args) == 2:
            try:
                n, m = int(args[0]), int(args[1])
                self.data = [[0] * m for _ in range(n)]
                self.rows = n
                self.cols = m
            except (TypeError, ValueError):
                raise ValueError("Enter (n, m) values or a list of lists.")

        elif len(args) == 1 and isinstance(args[0], list):
            data = args[0]

            if not all(isinstance(row, list) for row in data): ##list of lists
                raise ValueError("Input must be a list of lists.")

            col_lengths = [len(row) for row in data]
            if len(set(col_lengths)) != 1:
                raise ValueError(
                    f"All rows must have the same number of columns. "
                    f"Got row lengths: {col_lengths}"
                )

            self.data = [row[:] for row in data]
            self.rows = len(data)
            self.cols = col_lengths[0]

        else:
            raise ValueError("Enter (n, m) values or a list of lists.")

    def __getitem__(self, key):
        if isinstance(key, tuple):  ##Indexing M[i, j]
            i , j = key
            return self.data[i][j]
        else:                       ##returns the row so M[i][j] is valid
            return self.data[key]

    def __setitem__ (self, key, value):
        if isinstance(key, tuple):
            i , j = key
            self.data[i][j] = value
        else:
            self.data[key] = value

    ##M1 = M2
    def assign(self, other):
        if isinstance(other, matrix):
            if self.rows != other.rows or self.cols != other.cols:
                raise ValueError(
                    f"Size mismatch: cannot assign {other.rows}x{other.cols} "
                    f"to {self.rows}x{self.cols} matrix."
                )
            self.data = [row[:] for row in other.data]

        elif isinstance(other, list):
            if not all(isinstance(row, list) for row in other): ##list of lists
                raise ValueError("Input must be a list of lists.")
            if len(other) != self.rows or any(len(r) != self.cols for r in other):
                 raise ValueError(
                    f"Size mismatch: cannot assign {len(other)}x{len(other[0]) if other else 0} "
                    f"to {self.rows}x{self.cols} matrix."
                )
            self.data = [row[:] for row in other]

        else:
            raise TypeError("Can only assign from a matrix or list of lists.")

    def __repr__(self):
        rows_str = "\n".join(" [" + ", ".join(f"{v:6}" for v in row) + "]" for row in self.data)
        return f"matrix(\n{rows_str}\n) [{self.rows}x{self.cols}]"

In [7]:
##Testing

print("=" * 55)
print("TEST 1: Initialize with n, m")
M1 = matrix(4, 5)
print(M1)

TEST 1: Initialize with n, m
matrix(
 [     0,      0,      0,      0,      0]
 [     0,      0,      0,      0,      0]
 [     0,      0,      0,      0,      0]
 [     0,      0,      0,      0,      0]
) [4x5]


2. Add the following methods:
    * `shape()`: returns a tuple `(n,m)` of the shape of the matrix.
    * `transpose()`: returns a new matrix instance which is the transpose of the matrix.
    * `row(n)` and `column(n)`: that return the nth row or column of the matrix M as a new appropriately shaped matrix object.
    * `to_list()`: which returns the matrix as a list of lists.
    *  `block(n_0,n_1,m_0,m_1)` that returns a smaller matrix located at the n_0 to n_1 columns and m_0 to m_1 rows. 
    * Modify `__getitem__` implemented above to support slicing.
        

In [8]:
class matrix:
    def __init__(self, *args):
        if len(args) == 2:
            try:
                n, m = int(args[0]), int(args[1])
                self.data = [[0] * m for _ in range(n)]
                self.rows = n
                self.cols = m
            except (TypeError, ValueError):
                raise ValueError("Enter (n, m) values or a list of lists.")

        elif len(args) == 1 and isinstance(args[0], list):
            data = args[0]

            if not all(isinstance(row, list) for row in data): ##list of lists
                raise ValueError("Input must be a list of lists.")

            col_lengths = [len(row) for row in data]
            if len(set(col_lengths)) != 1:
                raise ValueError(
                    f"All rows must have the same number of columns. "
                    f"Got row lengths: {col_lengths}"
                )

            self.data = [row[:] for row in data]
            self.rows = len(data)
            self.cols = col_lengths[0]

        else:
            raise ValueError("Enter (n, m) values or a list of lists.")

    def __getitem__(self, key):
        if isinstance(key, tuple):
            row_key, col_key = key

            row_data = self.data[row_key] if isinstance(row_key, int) else self.data[row_key]

            if isinstance(row_key, int):
                if isinstance(col_key, int):
                    return row_data[col_key]
                else:
                    return matrix([row_data[col_key]])
            else:
                ##Multiple rows
                sliced_rows = [row[col_key] if isinstance(col_key, slice)
                               else [row[col_key]] for row in row_data]
                return matrix(sliced_rows)

        elif isinstance(key, slice):
            return matrix(self.data[key])

        else:
            return self.data[key]


    def __setitem__(self, key, value):
        if isinstance(key, tuple):
            i, j = key
            self.data[i][j] = value
        else:
            self.data[key] = value

    def assign(self, other):
        if isinstance(other, matrix):
            if self.rows != other.rows or self.cols != other.cols:
                raise ValueError(
                    f"Size mismatch: cannot assign {other.rows}x{other.cols} "
                    f"to {self.rows}x{self.cols} matrix."
                )
            self.data = [row[:] for row in other.data]

        elif isinstance(other, list):
            if not all(isinstance(row, list) for row in other):
                raise ValueError("Input must be a list of lists.")
            if len(other) != self.rows or any(len(r) != self.cols for r in other):
                raise ValueError(
                    f"Size mismatch: cannot assign {len(other)}x{len(other[0]) if other else 0} "
                    f"to {self.rows}x{self.cols} matrix."
                )
            self.data = [row[:] for row in other]

        else:
            raise TypeError("Can only assign from a matrix or list of lists.")

    def shape(self):
        return (self.rows, self.cols)


    def transpose(self):
        transposed = [[self.data[r][c] for r in range(self.rows)] for c in range(self.cols)]
        return matrix(transposed)


    def row(self, n):
        return matrix([self.data[n][:]]) ##returns nth row


    def column(self, n):
        return matrix([[self.data[r][n]] for r in range(self.rows)]) ##returns nth col


    def to_list(self):
        return [row[:] for row in self.data] ##returns matrix as list of lists


    def block(self, n_0, n_1, m_0, m_1):
        return matrix([row[m_0:m_1] for row in self.data[n_0:n_1]])

    def __repr__(self):
        rows_str = "\n".join(" [" + ", ".join(f"{v:6}" for v in row) + "]" for row in self.data)
        return f"matrix(\n{rows_str}\n) [{self.rows}x{self.cols}]"

In [9]:
M = matrix([[1, 2, 3, 4],
            [5, 6, 7, 8],
            [9, 10, 11, 12]])

print("=" * 55)
print("Testing: shape()")
print(M.shape())

print("\nTesting: transpose()")
print(M.transpose())

print("\nTesting: row(n)")
print(M.row(1))

print("\nTesting: column(n)")
print(M.column(2))

print("\nTesting: to_list()")
print(M.to_list())

print("\nTesting: Slicing with M[slice]")
print(M[0:2])

print("\nTesting: Slicing with M[slice, slice]")
print(M[0:2, 1:3])

print("\nTesting: Slicing with M[i, slice]")
print(M[1, 1:3])

print("\nTesting: Slicing with M[slice, j]")
print(M[0:3, 2])

Testing: shape()
(3, 4)

Testing: transpose()
matrix(
 [     1,      5,      9]
 [     2,      6,     10]
 [     3,      7,     11]
 [     4,      8,     12]
) [4x3]

Testing: row(n)
matrix(
 [     5,      6,      7,      8]
) [1x4]

Testing: column(n)
matrix(
 [     3]
 [     7]
 [    11]
) [3x1]

Testing: to_list()
[[1, 2, 3, 4], [5, 6, 7, 8], [9, 10, 11, 12]]

Testing: Slicing with M[slice]
matrix(
 [     1,      2,      3,      4]
 [     5,      6,      7,      8]
) [2x4]

Testing: Slicing with M[slice, slice]
matrix(
 [     2,      3]
 [     6,      7]
) [2x2]

Testing: Slicing with M[i, slice]
matrix(
 [     6,      7]
) [1x2]

Testing: Slicing with M[slice, j]
matrix(
 [     3]
 [     7]
 [    11]
) [3x1]


3. Write functions that create special matrices (note these are standalone functions, not member functions of your `matrix` class):
    * `constant(n,m,c)`: returns a `n` by `m` matrix filled with floats of value `c`.
    * `zeros(n,m)` and `ones(n,m)`: return `n` by `m` matrices filled with floats of value `0` and `1`, respectively.
    * `eye(n)`: returns the n by n identity matrix.

In [10]:
def constant(n, m, c):
    return matrix([[float(c)] * m for _ in range(n)])

def zeros(n, m):
    return constant(n, m, 0)

def ones(n, m):
    return constant(n, m, 1)

def eye(n):
    return matrix([[1.0 if r == c else 0.0 for c in range(n)] for r in range(n)])

In [12]:
print("=" * 55)
print("Testing: constants")
print(constant(4, 5, 8.1))

print("\nTesting: zeros")
print(zeros(3, 5))

print("\nTesting: ones")
print(ones(2, 3))

print("\nTesting: eye(4)")
print(eye(4))

print("\nTesting: eye(1) edge case")
print(eye(1))

print("\nTesting: constant with integer c is stored as float")
M = constant(1, 2, 3)
print(type(M[0, 0]))

Testing: constants
matrix(
 [   8.1,    8.1,    8.1,    8.1,    8.1]
 [   8.1,    8.1,    8.1,    8.1,    8.1]
 [   8.1,    8.1,    8.1,    8.1,    8.1]
 [   8.1,    8.1,    8.1,    8.1,    8.1]
) [4x5]

Testing: zeros
matrix(
 [   0.0,    0.0,    0.0,    0.0,    0.0]
 [   0.0,    0.0,    0.0,    0.0,    0.0]
 [   0.0,    0.0,    0.0,    0.0,    0.0]
) [3x5]

Testing: ones
matrix(
 [   1.0,    1.0,    1.0]
 [   1.0,    1.0,    1.0]
) [2x3]

Testing: eye(4)
matrix(
 [   1.0,    0.0,    0.0,    0.0]
 [   0.0,    1.0,    0.0,    0.0]
 [   0.0,    0.0,    1.0,    0.0]
 [   0.0,    0.0,    0.0,    1.0]
) [4x4]

Testing: eye(1) edge case
matrix(
 [   1.0]
) [1x1]

Testing: constant with integer c is stored as float
<class 'float'>


4. Add the following member functions to your class. Make sure to appropriately test the dimensions of the matrices to make sure the operations are correct.
    * `M.scalarmul(c)`: a matrix that is scalar product $cM$, where every element of $M$ is multiplied by $c$.
    * `M.add(N)`: adds two matrices $M$ and $N$. Don’t forget to test that the sizes of the matrices are compatible for this and all other operations.
    * `M.sub(N)`: subtracts two matrices $M$ and $N$.
    * `M.mat_mult(N)`: returns a matrix that is the matrix product of two matrices $M$ and $N$.
    * `M.element_mult(N)`: returns a matrix that is the element-wise product of two matrices $M$ and $N$.
    * `M.equals(N)`: returns true/false if $M==N$.

In [13]:
class matrix:
    def __init__(self, *args):
        if len(args) == 2:
            try:
                n, m = int(args[0]), int(args[1])
                self.data = [[0] * m for _ in range(n)]
                self.rows = n
                self.cols = m
            except (TypeError, ValueError):
                raise ValueError("Enter (n, m) values or a list of lists.")

        elif len(args) == 1 and isinstance(args[0], list):
            data = args[0]

            if not all(isinstance(row, list) for row in data): ##list of lists
                raise ValueError("Input must be a list of lists.")

            col_lengths = [len(row) for row in data]
            if len(set(col_lengths)) != 1:
                raise ValueError(
                    f"All rows must have the same number of columns. "
                    f"Got row lengths: {col_lengths}"
                )

            self.data = [row[:] for row in data]
            self.rows = len(data)
            self.cols = col_lengths[0]

        else:
            raise ValueError("Enter (n, m) values or a list of lists.")

    def __getitem__(self, key):
        if isinstance(key, tuple):
            row_key, col_key = key

            row_data = self.data[row_key] if isinstance(row_key, int) else self.data[row_key]

            if isinstance(row_key, int):
                if isinstance(col_key, int):
                    return row_data[col_key]
                else:
                    return matrix([row_data[col_key]])
            else:
                sliced_rows = [row[col_key] if isinstance(col_key, slice)
                               else [row[col_key]] for row in row_data]
                return matrix(sliced_rows)

        elif isinstance(key, slice):
            return matrix(self.data[key])

        else:
            return self.data[key]


    def __setitem__(self, key, value):
        if isinstance(key, tuple):
            i, j = key
            self.data[i][j] = value
        else:
            self.data[key] = value

    def assign(self, other):
        if isinstance(other, matrix):
            if self.rows != other.rows or self.cols != other.cols:
                raise ValueError(
                    f"Size mismatch: cannot assign {other.rows}x{other.cols} "
                    f"to {self.rows}x{self.cols} matrix."
                )
            self.data = [row[:] for row in other.data]

        elif isinstance(other, list):
            if not all(isinstance(row, list) for row in other):
                raise ValueError("Input must be a list of lists.")
            if len(other) != self.rows or any(len(r) != self.cols for r in other):
                raise ValueError(
                    f"Size mismatch: cannot assign {len(other)}x{len(other[0]) if other else 0} "
                    f"to {self.rows}x{self.cols} matrix."
                )
            self.data = [row[:] for row in other]

        else:
            raise TypeError("Can only assign from a matrix or list of lists.")

    def shape(self):
        return (self.rows, self.cols)

    def transpose(self):
        transposed = [[self.data[r][c] for r in range(self.rows)] for c in range(self.cols)]
        return matrix(transposed)

    def row(self, n):
        return matrix([self.data[n][:]])

    def column(self, n):
        return matrix([[self.data[r][n]] for r in range(self.rows)])

    def to_list(self):
        return [row[:] for row in self.data]

    def block(self, n_0, n_1, m_0, m_1):
        return matrix([row[m_0:m_1] for row in self.data[n_0:n_1]])

    def _check_same_size(self, N, op_name): ##check same size of 2 matrices
        if self.rows != N.rows or self.cols != N.cols:
            raise ValueError(
                f"{op_name} requires matrices of the same size. "
                f"Got {self.rows}x{self.cols} and {N.rows}x{N.cols}."
            )

    def scalarmul(self, c): ##returns a matrix in form of cM
        return matrix([[c * self.data[r][col] for col in range(self.cols)]
                       for r in range(self.rows)])

    def add(self, N): ##returns a matrix in form of M + N
        self._check_same_size(N, "Addition")
        return matrix([[self.data[r][c] + N.data[r][c] for c in range(self.cols)]
                       for r in range(self.rows)])

    def sub(self, N): ##returns a matrix in form of M - N
        self._check_same_size(N, "Subtraction")
        return matrix([[self.data[r][c] - N.data[r][c] for c in range(self.cols)]
                       for r in range(self.rows)])

    def element_mult(self, N): ##returns a matrix in form of M * N
        self._check_same_size(N, "Element-wise multiplication")
        return matrix([[self.data[r][c] * N.data[r][c] for c in range(self.cols)]
                       for r in range(self.rows)])

    def equals(self, N): ##returns true if M and N are ==
        if self.rows != N.rows or self.cols != N.cols:
            return False
        return all(self.data[r][c] == N.data[r][c]
                   for r in range(self.rows)
                   for c in range(self.cols))

    def __repr__(self):
        rows_str = "\n".join(" [" + ", ".join(f"{v:6}" for v in row) + "]" for row in self.data)
        return f"matrix(\n{rows_str}\n) [{self.rows}x{self.cols}]"

In [14]:
A = matrix([[1, 2, 3],
            [4, 5, 6]])

B = matrix([[7,  8,  9],
            [10, 11, 12]])

print("=" * 55)
print("Testing: scalarmul(4)")
print(A.scalarmul(4))

print("\nTesting: add(N)")
print(A.add(B))

print("\nTesting: sub(N)")
print(A.sub(B))

print("\nTesting: element_mult(N)")
print(A.element_mult(B))

print("\nTesting: equals(N)")
A_copy = matrix([[1, 2, 3], [4, 5, 6]]) ##true
print(A.equals(A_copy))

print("\nTesting: equals(N)")
print(A.equals(B)) ##false

print("\nTesting: equals(N)")
C = matrix([[1, 2], [3, 4]])
print(A.equals(C)) ##false

print("\nTesting: Size mismatch on add()")
try:
    A.add(C)
except ValueError as e:
    print(f"Caught expected error: {e}")

print("\nTesting: Size mismatch on sub()")
try:
    A.sub(C)
except ValueError as e:
    print(f"Caught expected error: {e}")

print("\nTesting: Size mismatch on element_mult()")
try:
    A.element_mult(C)
except ValueError as e:
    print(f"Caught expected error: {e}")

Testing: scalarmul(4)
matrix(
 [     4,      8,     12]
 [    16,     20,     24]
) [2x3]

Testing: add(N)
matrix(
 [     8,     10,     12]
 [    14,     16,     18]
) [2x3]

Testing: sub(N)
matrix(
 [    -6,     -6,     -6]
 [    -6,     -6,     -6]
) [2x3]

Testing: element_mult(N)
matrix(
 [     7,     16,     27]
 [    40,     55,     72]
) [2x3]

Testing: equals(N)
True

Testing: equals(N)
False

Testing: equals(N)
False

Testing: Size mismatch on add()
Caught expected error: Addition requires matrices of the same size. Got 2x3 and 2x2.

Testing: Size mismatch on sub()
Caught expected error: Subtraction requires matrices of the same size. Got 2x3 and 2x2.

Testing: Size mismatch on element_mult()
Caught expected error: Element-wise multiplication requires matrices of the same size. Got 2x3 and 2x2.


5. Overload python operators to appropriately use your functions in 4 and allow expressions like:
    * 2*M
    * M*2
    * M+N
    * M-N
    * M*N
    * M==N
    * M=N


In [16]:
class matrix:
    def __init__(self, *args):
        if len(args) == 2:
            try:
                n, m = int(args[0]), int(args[1])
                self.data = [[0] * m for _ in range(n)]
                self.rows = n
                self.cols = m
            except (TypeError, ValueError):
                raise ValueError("Enter (n, m) values or a list of lists.")

        elif len(args) == 1 and isinstance(args[0], list):
            data = args[0]

            if not all(isinstance(row, list) for row in data):
                raise ValueError("Input must be a list of lists.")

            col_lengths = [len(row) for row in data]
            if len(set(col_lengths)) != 1:
                raise ValueError(
                    f"All rows must have the same number of columns. "
                    f"Got row lengths: {col_lengths}"
                )

            self.data = [row[:] for row in data]
            self.rows = len(data)
            self.cols = col_lengths[0]

        else:
            raise ValueError("Enter (n, m) values or a list of lists.")

    def __getitem__(self, key):
        if isinstance(key, tuple):
            row_key, col_key = key

            row_data = self.data[row_key] if isinstance(row_key, int) else self.data[row_key]

            if isinstance(row_key, int):
                if isinstance(col_key, int):
                    return row_data[col_key]
                else:
                    return matrix([row_data[col_key]])
            else:
                sliced_rows = [row[col_key] if isinstance(col_key, slice)
                               else [row[col_key]] for row in row_data]
                return matrix(sliced_rows)

        elif isinstance(key, slice):
            return matrix(self.data[key])

        else:
            return self.data[key]


    def __setitem__(self, key, value):
        if isinstance(key, tuple):
            i, j = key
            self.data[i][j] = value
        else:
            self.data[key] = value

    def assign(self, other):
        if isinstance(other, matrix):
            if self.rows != other.rows or self.cols != other.cols:
                raise ValueError(
                    f"Size mismatch: cannot assign {other.rows}x{other.cols} "
                    f"to {self.rows}x{self.cols} matrix."
                )
            self.data = [row[:] for row in other.data]

        elif isinstance(other, list):
            if not all(isinstance(row, list) for row in other):
                raise ValueError("Input must be a list of lists.")
            if len(other) != self.rows or any(len(r) != self.cols for r in other):
                raise ValueError(
                    f"Size mismatch: cannot assign {len(other)}x{len(other[0]) if other else 0} "
                    f"to {self.rows}x{self.cols} matrix."
                )
            self.data = [row[:] for row in other]

        else:
            raise TypeError("Can only assign from a matrix or list of lists.")

    def shape(self):
        return (self.rows, self.cols)

    def transpose(self):
        transposed = [[self.data[r][c] for r in range(self.rows)] for c in range(self.cols)]
        return matrix(transposed)

    def row(self, n):
        return matrix([self.data[n][:]])

    def column(self, n):
        return matrix([[self.data[r][n]] for r in range(self.rows)])

    def to_list(self):
        return [row[:] for row in self.data]

    def block(self, n_0, n_1, m_0, m_1):
        return matrix([row[m_0:m_1] for row in self.data[n_0:n_1]])

    def _check_same_size(self, N, op_name):
        if self.rows != N.rows or self.cols != N.cols:
            raise ValueError(
                f"{op_name} requires matrices of the same size. "
                f"Got {self.rows}x{self.cols} and {N.rows}x{N.cols}."
            )

    def scalarmul(self, c):
        return matrix([[c * self.data[r][col] for col in range(self.cols)]
                       for r in range(self.rows)])

    def add(self, N):
        self._check_same_size(N, "Addition")
        return matrix([[self.data[r][c] + N.data[r][c] for c in range(self.cols)]
                       for r in range(self.rows)])

    def sub(self, N):
        self._check_same_size(N, "Subtraction")
        return matrix([[self.data[r][c] - N.data[r][c] for c in range(self.cols)]
                       for r in range(self.rows)])

    def element_mult(self, N):
        self._check_same_size(N, "Element-wise multiplication")
        return matrix([[self.data[r][c] * N.data[r][c] for c in range(self.cols)]
                       for r in range(self.rows)])

    def matmul(self, N):
        if self.cols != N.rows:
            raise ValueError(
                f"Matrix multiplication requires M.cols == N.rows. "
                f"Got {self.rows}x{self.cols} and {N.rows}x{N.cols}."
            )
        return matrix([[sum(self.data[r][k] * N.data[k][c] for k in range(self.cols))
                        for c in range(N.cols)]
                       for r in range(self.rows)])

    def equals(self, N):
        if self.rows != N.rows or self.cols != N.cols:
            return False
        return all(self.data[r][c] == N.data[r][c]
                   for r in range(self.rows)
                   for c in range(self.cols))

    def __add__(self, N):
        return self.add(N)

    def __sub__(self, N):
        return self.sub(N)

    def __mul__(self, other):
        if isinstance(other, matrix):
            return self.matmul(other)
        else:
            return self.scalarmul(other)

    def __rmul__(self, other):
        return self.scalarmul(other)

    def __eq__(self, N):
        return self.equals(N)

    def __ilshift__(self, other):
        self.assign(other)
        return self

    def __repr__(self):
        rows_str = "\n".join(" [" + ", ".join(f"{v:6}" for v in row) + "]" for row in self.data)
        return f"matrix(\n{rows_str}\n) [{self.rows}x{self.cols}]"

In [17]:
A = matrix([[1, 2, 3],
            [4, 5, 6]])

B = matrix([[7,  8,  9],
            [10, 11, 12]])

C = matrix([[1, 2],
            [3, 4],
            [5, 6]])

print("=" * 55)
print("Testing: M * c - scalar on right")
print(A * 5)

print("\nTesting: c * M - scalar on left")
print(5 * A)

print("\nTesting: M + N")
print(A + B)

print("\nTesting: M - N")
print(A - B)

print("\nTesting: M * N")
print(A * C)

print("\nTesting: M == N")
A_copy = matrix([[1, 2, 3], [4, 5, 6]])
print(A == A_copy)

print("\nTesting: M == N")
print(A == B)   ##False

print("\nTesting: M <<= N")
D = matrix(2, 3)
D <<= A
print(D) ##prints same values as A

print("\nTesting: M <<= list of lists")
E = matrix(2, 3)
E <<= [[9, 8, 7], [6, 5, 4]]
print(E)

print("\nTesting: Size mismatch on M * N")
try:
    print(A * B)
except ValueError as e:
    print(f"Caught expected error: {e}")

Testing: M * c - scalar on right
matrix(
 [     5,     10,     15]
 [    20,     25,     30]
) [2x3]

Testing: c * M - scalar on left
matrix(
 [     5,     10,     15]
 [    20,     25,     30]
) [2x3]

Testing: M + N
matrix(
 [     8,     10,     12]
 [    14,     16,     18]
) [2x3]

Testing: M - N
matrix(
 [    -6,     -6,     -6]
 [    -6,     -6,     -6]
) [2x3]

Testing: M * N
matrix(
 [    22,     28]
 [    49,     64]
) [2x2]

Testing: M == N
True

Testing: M == N
False

Testing: M <<= N
matrix(
 [     1,      2,      3]
 [     4,      5,      6]
) [2x3]

Testing: M <<= list of lists
matrix(
 [     9,      8,      7]
 [     6,      5,      4]
) [2x3]

Testing: Size mismatch on M * N
Caught expected error: Matrix multiplication requires M.cols == N.rows. Got 2x3 and 2x3.


6. Demonstrate the basic properties of matrices with your matrix class by creating two 2 by 2 example matrices using your Matrix class and illustrating the following:

$$
(AB)C=A(BC)
$$
$$
A(B+C)=AB+AC
$$
$$
AB\neq BA
$$
$$
AI=A
$$

In [20]:
A = matrix([[1, 2],
            [3, 4]])

B = matrix([[5, 6],
            [7, 8]])

C = matrix([[9,  10],
            [11, 12]])

I = eye(2)

print("=" * 55)
print("Matrices being used:")
print(f"\nA =\n{A}")
print(f"\nB =\n{B}")
print(f"\nC =\n{C}")
print(f"\nI (Identity) =\n{I}")


##(AB)C = A(BC)
print("\n" + "=" * 55)
print("Property 1: (AB)C = A(BC)")

AB_C  = (A * B) * C
A_BC  = A * (B * C)

print(f"\n(AB)C =\n{AB_C}")
print(f"\nA(BC) =\n{A_BC}")
print(f"\n(AB)C == A(BC): {AB_C == A_BC}")


##A(B+C) = AB + AC
print("\n" + "=" * 55)
print("Property 2: A(B+C) = AB + AC")

A_BplusC = A * (B + C)
AB_plus_AC = (A * B) + (A * C)

print(f"\nA(B+C) =\n{A_BplusC}")
print(f"\nAB + AC =\n{AB_plus_AC}")
print(f"\nA(B+C) == AB + AC: {A_BplusC == AB_plus_AC}")

##AB != BA
print("\n" + "=" * 55)
print("Property 3: AB != BA")

AB = A * B
BA = B * A

print(f"\nAB =\n{AB}")
print(f"\nBA =\n{BA}")
print(f"\nAB == BA: {AB == BA}")
print(f"AB != BA: {not (AB == BA)}")


## AI = A
print("\n" + "=" * 55)
print("Property 4: AI = A")

AI = A * I

print(f"\nA =\n{A}")
print(f"\nI =\n{I}")
print(f"\nAI =\n{AI}")
print(f"\nAI == A: {AI == A}")

Matrices being used:

A =
matrix(
 [     1,      2]
 [     3,      4]
) [2x2]

B =
matrix(
 [     5,      6]
 [     7,      8]
) [2x2]

C =
matrix(
 [     9,     10]
 [    11,     12]
) [2x2]

I (Identity) =
matrix(
 [   1.0,    0.0]
 [   0.0,    1.0]
) [2x2]

Property 1: (AB)C = A(BC)

(AB)C =
matrix(
 [   413,    454]
 [   937,   1030]
) [2x2]

A(BC) =
matrix(
 [   413,    454]
 [   937,   1030]
) [2x2]

(AB)C == A(BC): True

Property 2: A(B+C) = AB + AC

A(B+C) =
matrix(
 [    50,     56]
 [   114,    128]
) [2x2]

AB + AC =
matrix(
 [    50,     56]
 [   114,    128]
) [2x2]

A(B+C) == AB + AC: True

Property 3: AB != BA

AB =
matrix(
 [    19,     22]
 [    43,     50]
) [2x2]

BA =
matrix(
 [    23,     34]
 [    31,     46]
) [2x2]

AB == BA: False
AB != BA: True

Property 4: AI = A

A =
matrix(
 [     1,      2]
 [     3,      4]
) [2x2]

I =
matrix(
 [   1.0,    0.0]
 [   0.0,    1.0]
) [2x2]

AI =
matrix(
 [   1.0,    2.0]
 [   3.0,    4.0]
) [2x2]

AI == A: True


In [24]:
A = matrix([[1, 2],
            [3, 4]])

B = matrix([[5, 6],
            [7, 8]])

C = matrix([[9,  10],
            [11, 12]])

I = eye(2)

print("=" * 55)
print("Matrices being used:")
print(f"\nA =\n{A}")
print(f"\nB =\n{B}")
print(f"\nC =\n{C}")
print(f"\nI (Identity) =\n{I}")

print("\n" + "=" * 55)
print("Property 1: (AB)C = A(BC)")
AB_C  = (A * B) * C
A_BC  = A * (B * C)
print(f"\n(AB)C =\n{AB_C}")
print(f"\nA(BC) =\n{A_BC}")
print(f"\n(AB)C == A(BC): {AB_C == A_BC}")

print("\n" + "=" * 55)
print("Property 2: A(B+C) = AB + AC")
A_BplusC   = A * (B + C)
AB_plus_AC = (A * B) + (A * C)
print(f"\nA(B+C) =\n{A_BplusC}")
print(f"\nAB + AC =\n{AB_plus_AC}")
print(f"\nA(B+C) == AB + AC: {A_BplusC == AB_plus_AC}")

print("\n" + "=" * 55)
print("Property 3: AB != BA")
AB = A * B
BA = B * A
print(f"\nAB =\n{AB}")
print(f"\nBA =\n{BA}")
print(f"\nAB == BA: {AB == BA}")
print(f"AB != BA: {not (AB == BA)}")

print("\n" + "=" * 55)
print("Property 4: AI = A")
AI = A * I
print(f"\nA =\n{A}")
print(f"\nI =\n{I}")
print(f"\nAI =\n{AI}")
print(f"\nAI == A: {AI == A}")

Matrices being used:

A =
matrix(
 [     1,      2]
 [     3,      4]
) [2x2]

B =
matrix(
 [     5,      6]
 [     7,      8]
) [2x2]

C =
matrix(
 [     9,     10]
 [    11,     12]
) [2x2]

I (Identity) =
matrix(
 [   1.0,    0.0]
 [   0.0,    1.0]
) [2x2]

Property 1: (AB)C = A(BC)

(AB)C =
matrix(
 [   413,    454]
 [   937,   1030]
) [2x2]

A(BC) =
matrix(
 [   413,    454]
 [   937,   1030]
) [2x2]

(AB)C == A(BC): True

Property 2: A(B+C) = AB + AC

A(B+C) =
matrix(
 [    50,     56]
 [   114,    128]
) [2x2]

AB + AC =
matrix(
 [    50,     56]
 [   114,    128]
) [2x2]

A(B+C) == AB + AC: True

Property 3: AB != BA

AB =
matrix(
 [    19,     22]
 [    43,     50]
) [2x2]

BA =
matrix(
 [    23,     34]
 [    31,     46]
) [2x2]

AB == BA: False
AB != BA: True

Property 4: AI = A

A =
matrix(
 [     1,      2]
 [     3,      4]
) [2x2]

I =
matrix(
 [   1.0,    0.0]
 [   0.0,    1.0]
) [2x2]

AI =
matrix(
 [   1.0,    2.0]
 [   3.0,    4.0]
) [2x2]

AI == A: True
